In [5]:
import sqlite3
from arkindex_export import open_database, Element, Metadata, Transcription, database
from arkindex_export.queries import list_children
from pathlib import Path
from tqdm import tqdm
# pip install arkindex_export

DB_PATH = Path("sciencespo-archelec-20260217-121320.sqlite")


print(f"Le fichier existe : {DB_PATH.exists()}")
if DB_PATH.exists():
    print(f"Taille : {DB_PATH.stat().st_size / 1024 / 1024:.1f} MB")

   
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"\nTables dans la base :")
    for t in tables:
        print(f"  {t[0]}")
    conn.close()
else:
    print("FICHIER NON TROUVÉ !")
    print(f"Répertoire courant : {Path.cwd()}")
    print(f"Fichiers ici : {list(Path('.').glob('*'))}")



Le fichier existe : False
FICHIER NON TROUVÉ !
Répertoire courant : /home/onyxia/work/ensae-nlp-archelec/notebooks
Fichiers ici : [PosixPath('02_exploration.ipynb'), PosixPath('election_download.ipynb'), PosixPath('.ipynb_checkpoints'), PosixPath('01_data_loading.ipynb.ipynb')]


In [1]:

def index_database(db_path: Path):
    """Add performance indexes to an Arkindex SQLite export."""
    open_database(db_path)
    if database.is_closed():
        database.connect()

    with database.atomic():
        # Index seulement sur element.type (table qui existe à coup sûr)
        database.execute_sql("""
            CREATE INDEX IF NOT EXISTS idx_element_type
            ON element(type);
        """)

    database.close()
    print("Indexing completed.")


index_database(DB_PATH)
open_database(DB_PATH)

folders = Element.select().where(Element.type == 'folder')
print(f"Nombre total de dossiers : {folders.count()}\n")
for folder in folders:
    print(f"  {folder.name}  ->  {folder.id}")

YEARS = ['1958', '1962', '1967', '1968', '1973', '1978', '1981', '1988', '1993']
ELECTIONS = ['legislatives', 'presidentielle']

folder_id = {year: {} for year in YEARS}


folder_id['1981']['legislatives'] = 'd51ea3db-68ee-4cc0-a87f-736ee17c5f87'
folder_id['1988']['legislatives'] = 'dfba9f5c-02de-478c-85c5-0ee780455433'
folder_id['1993']['legislatives'] = 'cf29300f-40bf-4b61-be93-6cb631be8fab'
folder_id['1988']['presidentielle'] = 'fd5bee0a-83e8-4bdc-aa48-52331af2e151'
folder_id['1958']['legislatives'] = 'ace400a7-a191-4796-848b-1c3644f7fbbf'
folder_id['1962']['legislatives'] = '3dda21f1-186d-4a0c-aaa9-62384bf35c60'
folder_id['1967']['legislatives'] = '80659501-401f-4642-a537-d2241e5d3d87'
folder_id['1968']['legislatives'] = '2df5445d-6a85-405d-8824-f25c673588f2'
folder_id['1973']['legislatives'] = '929ef8f9-c2c9-4d87-b858-f7af790aa752'
folder_id['1978']['legislatives'] = '117d3883-f985-476e-b8d4-3732d8753d7a'

TEXT_FOLDER = "text_files"
output_folder = Path(TEXT_FOLDER)
output_folder.mkdir(exist_ok=True)

for year in YEARS:
    for e_type in ELECTIONS:
        (output_folder / year / e_type).mkdir(parents=True, exist_ok=True)


print("\nNumber of folders:", Element.select().where(Element.type == 'folder').count())
print("Number of pages:", Element.select().where(Element.type == 'page').count())


for year in YEARS:
    for e_type in ELECTIONS:
        f_id = folder_id[year].get(e_type, None)
        if not f_id:
            continue

        print(f"\n--- {year} / {e_type} ---")
        documents = list_children(f_id).where(Element.type == 'document')
        print(f"  Documents trouvés : {documents.count()}")

        transcriptions_number = 0
        for document in tqdm(documents, desc=f"{year}/{e_type}"):
            pages = list_children(document.id).where(Element.type == 'page')
            transcriptions = ""
            for page in pages:
                page_transcription = (
                    Transcription.select()
                    .where(Transcription.element == page.id)
                    .first()
                )
                if page_transcription:
                    transcriptions += page_transcription.text

            if transcriptions:
                out_path = f"{TEXT_FOLDER}/{year}/{e_type}/{document.name}.txt"
                with open(out_path, "w", encoding="utf-8") as f:
                    f.write(transcriptions)
                transcriptions_number += 1

        print(f"  Transcriptions extraites : {transcriptions_number}")

NameError: name 'Path' is not defined

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 83.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for peewee: filename=peewee-3.17.0-py3-none-any.whl size=135807 sha256=145c16e3ba23f6ea577cdeab545d7f8c9e9e5a2900d3e01a4664ba5c17720ac9
  Stored in directory: /home/onyxia/.cache/pip/wheels/36/e0/cf/19a6133187b80ef6d769978c73839e185187d71308e666aa1a
Successfully built peewee
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [arkindex_export]
Note: you may need to restart the kernel to use updated packages.
